In [1]:
import earthaccess
earthaccess.login(strategy="interactive")

In [ ]:
from pathlib import Path
import sys

_REPO_ROOT = Path("..").resolve()
_SRC = _REPO_ROOT / "src"
if _SRC.is_dir() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from critical_minerals_aster.config import load_site_config
from critical_minerals_aster.paths import site_paths_for

site = load_site_config(_REPO_ROOT / "sites" / "mcdermitt.yaml")
paths = site_paths_for(site, _REPO_ROOT)
bbox = site.bbox_wgs84

results = earthaccess.search_data(
    short_name="AST_L1T",
    bounding_box=bbox,
    temporal=(site.temporal_start, site.temporal_end),
    count=10,
)

print(f"Site: {site.name} ({site.id})")
print(f"Found {len(results)} granules")
for r in results:
    print(r)

Found 10 granules
Collection: {'ShortName': 'AST_L1T', 'Version': '004'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Longitude': -118.604755515461, 'Latitude': 41.7997904766487}, {'Longitude': -118.604755515461, 'Latitude': 41.1066916977236}, {'Longitude': -117.573696231509, 'Latitude': 41.1066916977236}, {'Longitude': -117.573696231509, 'Latitude': 41.7997904766487}, {'Longitude': -118.604755515461, 'Latitude': 41.7997904766487}]}}]}}}
Temporal coverage: {'SingleDateTime': '2010-03-27T05:49:47.017000Z'}
Size(MB): 3.964522361755371
Data: ['https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/AST_L1T.004/AST_L1T_00403272010054947_20250703101203/AST_L1T_00403272010054947_20250703101203_TIR.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/AST_L1T.004/AST_L1T_00403272010054947_20250703101203/AST_L1T_00403272010054947_20250703101203_TIR_B10.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protect

/opt/homebrew/Caskroom/miniforge/base/envs/aster-minerals/lib/python3.11/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/opt/homebrew/Caskroom/miniforge/base/envs/aster-minerals/lib/python3.11/site-packages/earthaccess/results.py:364: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  Size(MB): {self.size()}


In [3]:
from critical_minerals_aster.spectral import (
    extract_granule_id,
    score_granule,
    select_granule,
)

print("Granule scores (coverage × 10 + TIR band count):")
for i, r in enumerate(results):
    try:
        gid = extract_granule_id(r)
        cov, composite, bands = score_granule(r, bbox)
        print(f"  [{i}] {gid} | coverage={cov:.2f} | TIR bands={bands}/5 | score={composite:.2f}")
    except (ValueError, KeyError, IndexError, TypeError) as exc:
        print(f"  [{i}] (skipped: {exc})")

target = select_granule(results, bbox, granule_id_override=site.granule_id)
selected_id = extract_granule_id(target)
cov, composite, bands = score_granule(target, bbox)
print(f"\nSelected: {selected_id}")
print(f"  override={site.granule_id!r} | coverage={cov:.2f} | TIR bands={bands}/5 | score={composite:.2f}")

2010-03-27 | VNIR: False | SWIR: False | 4.0 MB
2010-03-27 | VNIR: False | SWIR: False | 4.3 MB
2010-03-27 | VNIR: False | SWIR: False | 4.3 MB
2010-07-01 | VNIR: False | SWIR: False | 4.4 MB
2010-07-01 | VNIR: False | SWIR: False | 4.7 MB
2010-07-01 | VNIR: False | SWIR: False | 4.6 MB
2010-07-23 | VNIR: True | SWIR: False | 94.9 MB
2010-07-23 | VNIR: True | SWIR: False | 96.9 MB
2010-07-23 | VNIR: True | SWIR: False | 111.5 MB
2011-03-20 | VNIR: True | SWIR: False | 56.6 MB

0 granules with SWIR bands


/var/folders/x4/28w1j1bj1qs7p88q3dc5snyw0000gn/T/ipykernel_42598/1240846317.py:7: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  print(f"{r['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime'] if 'RangeDateTime' in r['umm']['TemporalExtent'] else r['umm']['TemporalExtent']['SingleDateTime'][:10]} | VNIR: {has_vnir} | SWIR: {has_swir} | {r.size():.1f} MB")


In [4]:
# Data links for the selected granule
for url in target.data_links():
    print(url)

https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/AST_L1T.004/AST_L1T_00407232010184954_20250705074032/AST_L1T_00407232010184954_20250705074032_TIR.tif
https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/AST_L1T.004/AST_L1T_00407232010184954_20250705074032/AST_L1T_00407232010184954_20250705074032_TIR_B10.tif
https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/AST_L1T.004/AST_L1T_00407232010184954_20250705074032/AST_L1T_00407232010184954_20250705074032_TIR_B11.tif
https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/AST_L1T.004/AST_L1T_00407232010184954_20250705074032/AST_L1T_00407232010184954_20250705074032_TIR_B12.tif
https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/AST_L1T.004/AST_L1T_00407232010184954_20250705074032/AST_L1T_00407232010184954_20250705074032_TIR_B13.tif
https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/AST_L1T.004/AST_L1T_00407232010184954_20250705074032/AST_L1T_00407232010184954_20250705074032_TIR_B14.

In [5]:
paths.aster_dir.mkdir(parents=True, exist_ok=True)
files = earthaccess.download(target, str(paths.aster_dir))
print(f"Downloaded {len(files)} files to {paths.aster_dir}:")
for f in files:
    print(f)

/opt/homebrew/Caskroom/miniforge/base/envs/aster-minerals/lib/python3.11/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded:
../data/aster/AST_L1T_00407232010184954_20250705074032_TIR.tif
../data/aster/AST_L1T_00407232010184954_20250705074032_TIR_B10.tif
../data/aster/AST_L1T_00407232010184954_20250705074032_TIR_B11.tif
../data/aster/AST_L1T_00407232010184954_20250705074032_TIR_B12.tif
../data/aster/AST_L1T_00407232010184954_20250705074032_TIR_B13.tif
../data/aster/AST_L1T_00407232010184954_20250705074032_TIR_B14.tif
../data/aster/AST_L1T_00407232010184954_20250705074032_VNIR_B01.tif
../data/aster/AST_L1T_00407232010184954_20250705074032_VNIR_B02.tif
../data/aster/AST_L1T_00407232010184954_20250705074032_VNIR_B03N.tif
../data/aster/AST_L1T_00407232010184954_20250705074032_VNIR.tif


In [6]:
import rasterio

for f in sorted(paths.aster_dir.iterdir()):
    if f.suffix == ".tif":
        with rasterio.open(f) as src:
            print(f"{f.name}")
            print(f"  CRS: {src.crs}")
            print(f"  Shape: {src.shape}")
            print(f"  Bands: {src.count}")
            print(f"  Dtype: {src.dtypes[0]}")

AST_L1T_00407232010184954_20250705074032_TIR.tif
  CRS: EPSG:32611
  Shape: (853, 955)
  Bands: 3
  Dtype: uint8
AST_L1T_00407232010184954_20250705074032_TIR_B10.tif
  CRS: EPSG:32611
  Shape: (852, 954)
  Bands: 1
  Dtype: uint16
AST_L1T_00407232010184954_20250705074032_TIR_B11.tif
  CRS: EPSG:32611
  Shape: (852, 954)
  Bands: 1
  Dtype: uint16
AST_L1T_00407232010184954_20250705074032_TIR_B12.tif
  CRS: EPSG:32611
  Shape: (852, 954)
  Bands: 1
  Dtype: uint16
AST_L1T_00407232010184954_20250705074032_TIR_B13.tif
  CRS: EPSG:32611
  Shape: (852, 954)
  Bands: 1
  Dtype: uint16
AST_L1T_00407232010184954_20250705074032_TIR_B14.tif
  CRS: EPSG:32611
  Shape: (852, 954)
  Bands: 1
  Dtype: uint16
AST_L1T_00407232010184954_20250705074032_VNIR.tif
  CRS: EPSG:32611
  Shape: (5113, 5725)
  Bands: 3
  Dtype: uint8
AST_L1T_00407232010184954_20250705074032_VNIR_B01.tif
  CRS: EPSG:32611
  Shape: (5112, 5724)
  Bands: 1
  Dtype: uint8
AST_L1T_00407232010184954_20250705074032_VNIR_B02.tif
  CRS: 

In [7]:
# Footprint vs study bbox for the selected granule
pts = target["umm"]["SpatialExtent"]["HorizontalSpatialDomain"]["Geometry"]["GPolygons"][0]["Boundary"]["Points"]
lats = [float(p["Latitude"]) for p in pts]
lons = [float(p["Longitude"]) for p in pts]
date = target["umm"]["TemporalExtent"]["SingleDateTime"][:10]
print(f"Selected scene date: {date}")
print(f"  Footprint lat {min(lats):.2f}–{max(lats):.2f} | lon {min(lons):.2f}–{max(lons):.2f}")
print(f"  Study bbox lat {bbox[1]:.2f}–{bbox[3]:.2f} | lon {bbox[0]:.2f}–{bbox[2]:.2f}")
print(f"  Granule id: {selected_id}")

[0] 2010-03-27 | lat 41.11-41.80 | lon -118.60--117.57 | 4 MB
[1] 2010-03-27 | lat 41.63-42.33 | lon -118.79--117.75 | 4 MB
[2] 2010-03-27 | lat 42.16-42.85 | lon -118.98--117.93 | 4 MB
[3] 2010-07-01 | lat 41.11-41.80 | lon -118.60--117.56 | 4 MB
[4] 2010-07-01 | lat 41.63-42.33 | lon -118.78--117.74 | 5 MB
[5] 2010-07-01 | lat 42.16-42.86 | lon -118.97--117.92 | 5 MB
[6] 2010-07-23 | lat 42.28-42.97 | lon -117.89--116.83 | 95 MB
[7] 2010-07-23 | lat 41.75-42.44 | lon -118.06--117.02 | 97 MB
[8] 2010-07-23 | lat 41.22-41.92 | lon -118.23--117.19 | 111 MB
[9] 2011-03-20 | lat 41.86-42.56 | lon -118.90--117.85 | 57 MB


/var/folders/x4/28w1j1bj1qs7p88q3dc5snyw0000gn/T/ipykernel_42598/3466748412.py:10: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  print(f"[{i}] {date} | lat {min(lats):.2f}-{max(lats):.2f} | lon {min(lons):.2f}-{max(lons):.2f} | {r.size():.0f} MB")


In [8]:
# Analysis granule is selected above; re-run select_granule without override for auto-pick at new sites:
# target = select_granule(results, bbox, granule_id_override=None)

/opt/homebrew/Caskroom/miniforge/base/envs/aster-minerals/lib/python3.11/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/10 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded:
../data/aster/AST_L1T_00407232010184946_20250705074029_TIR.tif
../data/aster/AST_L1T_00407232010184946_20250705074029_TIR_B10.tif
../data/aster/AST_L1T_00407232010184946_20250705074029_TIR_B11.tif
../data/aster/AST_L1T_00407232010184946_20250705074029_TIR_B12.tif
../data/aster/AST_L1T_00407232010184946_20250705074029_TIR_B13.tif
../data/aster/AST_L1T_00407232010184946_20250705074029_TIR_B14.tif
../data/aster/AST_L1T_00407232010184946_20250705074029_VNIR_B01.tif
../data/aster/AST_L1T_00407232010184946_20250705074029_VNIR_B02.tif
../data/aster/AST_L1T_00407232010184946_20250705074029_VNIR_B03N.tif
../data/aster/AST_L1T_00407232010184946_20250705074029_VNIR.tif
